# 🦙 Local Model Server Quickstart & Diagnostics

This notebook guides you through connecting to your local or LAN-hosted model server (`Ollama` / `llama-server`) using Python and the OpenAI SDK.

### Features Covered:
1. **Server Health & Diagnostics**: Check root connectivity and discover available models via `client.models.list()`.
2. **OpenAI-Compatible Client**: Initialize the official `openai` Python SDK targeting your local endpoint.
3. **Chat Completions & Streaming**: Run prompt completions, chat interactions, and real-time token streaming.
4. **Structured Outputs**: Request JSON formatted output from the model.


In [ ]:
# Ensure project root is in sys.path and load environment variables
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
print(f"Project root: {project_root}")
print(f"Server Host: {os.getenv('LLAMA_SERVER_HOST', 'http://localhost:11434')}")
print(f"API Base:    {os.getenv('LLAMA_API_BASE', 'http://localhost:11434/v1')}")
print(f"Chat Model:  {os.getenv('DEFAULT_CHAT_MODEL', 'qwen2.5-coder:7b')}")


## 1. Check Server Status & Inspect Model Metadata

We test server connectivity and retrieve model metadata (context length, parameter size, quantization, etc.).


In [ ]:
from src.llama_helper import check_server_health, get_server_props
from rich.pretty import pprint

health = check_server_health()
print(f"Server Health Status: {health.get('status')}")
pprint(health)

props = get_server_props()
print("\nServer Properties & Model Info:")
pprint(props)


## 2. Initialize OpenAI Client

The server exposes a drop-in `/v1` OpenAI-compatible REST API. We configure the standard `OpenAI` client pointing to `LLAMA_API_BASE`.


In [ ]:
from src.llama_helper import get_openai_client, DEFAULT_CHAT_MODEL

client = get_openai_client()
print(f"OpenAI Client base_url: {client.base_url}")
print(f"Default model:          {DEFAULT_CHAT_MODEL}")

# Query models from the server
models = client.models.list()
print("\nDiscovered models on server:")
for m in models.data:
    print(f" • {m.id}")


## 3. Basic Chat Completion

Send a standard chat completion request with system and user messages.


In [ ]:
try:
    response = client.chat.completions.create(
        model=DEFAULT_CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You are a concise AI assistant. Explain concepts clearly and briefly."},
            {"role": "user", "content": "What is the primary difference between dense vector embeddings and sparse representations?"}
        ],
        temperature=0.7,
        max_tokens=150
    )
    print("--- Response ---")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"Request failed: {e}\n(Ensure llama-server is running with: llama-server -m <model.gguf> --port 11434)")


## 4. Streaming Responses (Real-Time Token Output)

Stream tokens progressively as `llama-server` generates them.


In [ ]:
prompt = "Write a 4-line poem about linear algebra, high dimensions, and vectors."

try:
    stream = client.chat.completions.create(
        model=DEFAULT_CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        temperature=0.7,
        max_tokens=120
    )
    print("--- Streaming Output ---")
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            print(delta, end="", flush=True)
    print()
except Exception as e:
    print(f"Streaming failed: {e}")


## 5. Structured JSON Output

Local llama models can return structured JSON outputs for extraction tasks.


In [ ]:
try:
    response = client.chat.completions.create(
        model=DEFAULT_CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You are a metadata extractor. Respond with a JSON object containing keys: 'term', 'definition', 'use_cases'."},
            {"role": "user", "content": "Define 'Cosine Similarity'."}
        ],
        response_format={"type": "json_object"},
        temperature=0.2,
        max_tokens=200
    )
    print("--- Structured JSON Response ---")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"Structured output test note: {e}")
